# Direct training inputs: series-safe 384×384

Attach the 32 private `rsna-series-safe-384-shard-*` datasets. The preview is a separate eight-study sample and must not be attached alongside production shards. Images are already normalized float16 tensors; no crop, resize, /255 or normalization is needed. Gold74 is excluded by the train split; HYBRID targets are provided separately. This notebook only checks and loads data.


In [ ]:
"""Load direct-384 preprocessed images without any pixel transformation."""
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset


class SeriesSafe384Dataset(Dataset):
    def __init__(self,roots,split='train',k=None,labels=True):
        roots=[Path(p) for p in roots]
        recipes=[json.loads((p/'recipe.json').read_text()) for p in roots]
        if not recipes or any(r['format']!='rsna_series_safe_384_v1' for r in recipes) or len({r['sha256'] for r in recipes})!=1:
            raise ValueError('Attach only the new series-safe direct-384 recipe')
        if k is not None and (not isinstance(k,int) or k<1):raise ValueError('K must be positive or None')
        splits=pd.concat([pd.read_csv(p/'splits.csv',dtype={'StudyInstanceUID':str}) for p in roots],ignore_index=True)
        if splits.StudyInstanceUID.duplicated().any():raise ValueError('Duplicate shards/study IDs')
        wanted=set(splits.loc[splits['split']==split,'StudyInstanceUID'])
        self.files=sorted(p for root in roots for p in root.rglob('*.npz') if p.stem in wanted)
        if {p.stem for p in self.files}!=wanted or len(self.files)!=len(wanted):raise ValueError('Missing or duplicate study files')
        self.k=k;self.labels=None
        if labels:
            if split!='train':raise ValueError('Gold labels remain separate; use labels=False for evaluation images')
            table=pd.concat([pd.read_csv(p/'labels_hybrid_train.csv',dtype={'StudyInstanceUID':str}) for p in roots],ignore_index=True)
            findings=['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
            if table.StudyInstanceUID.duplicated().any() or wanted!=set(table.StudyInstanceUID) or not np.isfinite(table[findings].to_numpy()).all():
                raise ValueError('Invalid HYBRID training labels')
            self.labels=table.set_index('StudyInstanceUID')[findings]

    def __len__(self):return len(self.files)

    def __getitem__(self,index):
        file=self.files[index]
        with np.load(file,allow_pickle=False) as saved:
            images=torch.from_numpy(saved['images'])
            series=saved['triplet_series_uids']
            indices=saved['triplet_indices']
            owners=saved['slice_series_uids'][indices]
            if not np.all(owners==owners[:,:1]) or not np.all(owners[:,0]==series) or not np.all(saved['same_series']):
                raise ValueError('Cross-series triplet detected')
            if images.dtype!=torch.float16 or images.ndim!=4 or tuple(images.shape[1:])!=(3,384,384) or not len(images):
                raise ValueError('Invalid preprocessed images')
            slots=torch.from_numpy(saved['triplet_slot_ids'])
            present=torch.from_numpy(saved['slot_present'])
        if self.k is not None:
            chosen=torch.randperm(len(images))[:self.k]
            if len(chosen)<self.k:chosen=torch.cat([chosen,torch.randint(len(images),(self.k-len(chosen),))])
            images,slots=images[chosen],slots[chosen]
        item=dict(images=images,triplet_slot_ids=slots,slot_present=present,study_uid=file.stem)
        if self.labels is not None:item['targets']=torch.tensor(self.labels.loc[file.stem].to_numpy(dtype=np.float32))
        return item


In [ ]:
roots=sorted({p.parent for p in Path('/kaggle/input').rglob('recipe.json')
    if 'rsna-series-safe-384-shard-' in str(p)})
assert roots, 'Attach the new series-safe 384 dataset shards'
train=SeriesSafe384Dataset(roots,split='train',k=None,labels=True)
gold=SeriesSafe384Dataset(roots,split='held_out_gold',k=None,labels=False)
assert not {p.stem for p in train.files}&{p.stem for p in gold.files}
if len(roots)==32:
    assert len(train)==4333 and len(gold)==74
print({'attached_shards':len(roots),'training_studies':len(train),'gold_studies':len(gold)})
sample=train[0]
print(sample['study_uid'],sample['images'].shape,sample['images'].dtype)
print('These images are ready to pass to the model.')
import hashlib
rows=[]
for root in roots:
    manifest=json.loads((root/'manifest.json').read_text())
    assert manifest['complete'] and not manifest['failures']
    rows.extend(manifest['completed'])
    for row in manifest['completed']:
        file=root/row['file']
        assert file.is_file() and file.stat().st_size==row['bytes']
        assert row['cross_series_windows']==0 and row['source_slices']<=72 and row['windows']<=60
    # Independently exercise one study from each attached shard without any
    # pixel transformation, and verify ownership and the published checksum.
    row=manifest['completed'][0];file=root/row['file']
    with file.open('rb') as stream: assert hashlib.file_digest(stream,'sha256').hexdigest()==row['sha256']
    with np.load(file,allow_pickle=False) as saved:
        owners=saved['slice_series_uids'][saved['triplet_indices']]
        assert np.all(owners==owners[:,:1]) and np.all(saved['same_series'])
        assert np.all(owners[:,0]==saved['triplet_series_uids'])
        assert saved['images'].dtype==np.float16 and saved['images'].shape[1:]==(3,384,384)
        assert np.isfinite(saved['images']).all()
assert len({r['study_uid'] for r in rows})==len(rows)
summary=dict(attached_shards=len(roots),studies=len(rows),training_studies=len(train),gold_studies=len(gold),
    source_slices=sum(r['source_slices'] for r in rows),triplets=sum(r['windows'] for r in rows),
    cross_series_triplets=0,npz_bytes=sum(r['bytes'] for r in rows),
    recipe_sha256=json.loads((roots[0]/'recipe.json').read_text())['sha256'])
Path('/kaggle/working/dataset_validation.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))


`k=None` returns every triplet. Study bags have variable sizes; use batch size one or a ragged collate. Convert tensor dtype only if your model requires float32. Preserve whole triplets when batching.
